In [ ]:
from glob import glob
import os

import pandas as pd

In [ ]:
DATASETS_BASE_PATH = "../preprocessing/evaluation/datasets"
SHOTS_BASE_PATH = "./shots"
SPLITS_BASE_PATH = "./splits"

In [ ]:
dataset_name = "smell_doc"

In [ ]:
try:
    # logger.info("Loading the %s dataset...", params.dataset_name)
    data_df = pd.read_parquet(
        os.path.join(DATASETS_BASE_PATH, f"{dataset_name}.parquet")
    )

    splits_filename = f"{dataset_name}.*.*.csv"
    # logger.info("Loading the splits file named %s...", splits_filename)
    splits_df = pd.read_csv(glob(os.path.join(SPLITS_BASE_PATH, splits_filename))[0])
except FileNotFoundError as e:
    # logger.error(e)
    # sys.exit(1)
    pass

In [ ]:
# logger.info("Selecting the data for testing...")
shots_df = data_df.loc[splits_df["fold_1"] != 2]
test_df = data_df.loc[splits_df["fold_1"] == 2]
# logger.info("Number of instances for testing: %d", test_df.shape[0])
print("Number of instances for shot selection:", shots_df.shape[0])
print("Number of instances for testing:", test_df.shape[0])

In [ ]:
shots_df["len"] = shots_df["text"].str.len()

In [ ]:
stats = shots_df["len"].describe(percentiles=[.1, .9])
stats

In [ ]:
shots_reduced_df = shots_df.loc[(shots_df["len"] > stats["10%"].item()) & (shots_df["len"] < stats["90%"].item())]

In [ ]:
if "label" in shots_df.columns:
    n_labels = shots_df["label"].nunique()
else:
    n_labels = len([c for c in shots_df.columns if "label_" in c])

n_labels

In [ ]:
all_shots = []

if "label" in shots_df.columns:
    for _, row in test_df.iterrows():
        shot_ids = shots_reduced_df.groupby("label", group_keys=False).apply(lambda x: x.sample(n_labels, replace=True)).sample(3)["id"].to_list()
        all_shots.append(shot_ids)
else:
    for _, row in test_df.iterrows():
        shot_ids = shots_reduced_df.groupby([c for c in shots_df.columns if "label_" in c], group_keys=False).apply(lambda x: x.sample(n_labels, replace=True)).sample(3)["id"].to_list()
        all_shots.append(shot_ids)

In [ ]:
all_shots_df = pd.DataFrame(all_shots, index=test_df.index)

In [ ]:
os.makedirs(SHOTS_BASE_PATH, exist_ok=True)

In [ ]:
all_shots_df.to_csv(os.path.join(SHOTS_BASE_PATH, f"{dataset_name}.csv"))

In [ ]:
pd.read_csv(os.path.join(SHOTS_BASE_PATH, f"{dataset_name}.csv"), index_col=0)